# 🦸 Real-Time AI Superhero Visual Effects System
> Cinematic Avengers-style effects powered by MediaPipe + OpenCV — No API Key Required

### Features:
- 🔵 **Iron Man** — Energy blasts from palms (hands raised high)
- ⚡ **Thor** — Lightning arcs from wrist (arms wide open)
- 🛡️ **Captain America** — Force shield overlay (arms crossed)
- 💚 **Hulk Smash** — Shockwave ground-pound effect (both hands low)
- 🕷️ **Spider-Man** — Web-shooting trail (one hand forward)
- 📸 **Capture** — Press `S` to screenshot, `R` to record, `Q` to quit

## Step 1: Install Dependencies

In [8]:
import sys
print(sys.version)

3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]


In [9]:
!pip install opencv-python mediapipe numpy Pillow

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


## Step 2: Imports & Configuration

In [10]:
import cv2
import mediapipe as mp
import numpy as np
import time
import os
import math
import random
from datetime import datetime

# Output directories
os.makedirs('screenshots', exist_ok=True)
os.makedirs('recordings', exist_ok=True)

print('✅ All imports successful!')
print('📁 Output folders: screenshots/ and recordings/')

✅ All imports successful!
📁 Output folders: screenshots/ and recordings/


## Step 3: Visual Effects Engine

In [11]:
# ─────────────────────────────────────────────
#  PARTICLE SYSTEM
# ─────────────────────────────────────────────
class Particle:
    def __init__(self, x, y, color, speed=5, life=30, size=4, vx=None, vy=None):
        self.x = float(x)
        self.y = float(y)
        self.color = color
        self.life = life
        self.max_life = life
        self.size = size
        angle = random.uniform(0, 2 * math.pi)
        self.vx = vx if vx is not None else math.cos(angle) * speed
        self.vy = vy if vy is not None else math.sin(angle) * speed
        self.gravity = 0.15

    def update(self):
        self.x += self.vx
        self.y += self.vy
        self.vy += self.gravity
        self.life -= 1

    def draw(self, frame):
        alpha = self.life / self.max_life
        radius = max(1, int(self.size * alpha))
        color = tuple(int(c * alpha) for c in self.color)
        cx, cy = int(self.x), int(self.y)
        if 0 < cx < frame.shape[1] and 0 < cy < frame.shape[0]:
            # Glow: draw larger, dimmer circle first
            glow_color = tuple(int(c * alpha * 0.4) for c in self.color)
            cv2.circle(frame, (cx, cy), radius * 2, glow_color, -1)
            cv2.circle(frame, (cx, cy), radius, color, -1)


# ─────────────────────────────────────────────
#  FX RENDERER
# ─────────────────────────────────────────────
class SuperheroFX:
    def __init__(self):
        self.particles = []
        self.trail_points = []     # Spider-Man web trail
        self.lightning_cache = []
        self.frame_count = 0

    def _add_particles(self, x, y, color, n=15, speed=6, life=40, size=5,
                       directed=False, dx=0, dy=-1):
        for _ in range(n):
            if directed:
                spread = 0.8
                vx = dx * speed + random.uniform(-spread, spread) * speed * 0.3
                vy = dy * speed + random.uniform(-spread, spread) * speed * 0.3
                self.particles.append(
                    Particle(x, y, color, speed, life, size, vx=vx, vy=vy))
            else:
                self.particles.append(
                    Particle(x, y, color, speed, life, size))

    # ── Iron Man Energy Blast ──
    def iron_man_blast(self, frame, lm, w, h):
        palm_ids = [mp.solutions.pose.PoseLandmark.LEFT_WRIST,
                    mp.solutions.pose.PoseLandmark.RIGHT_WRIST]
        colors = [(0, 140, 255), (0, 200, 255)]  # Orange-blue arc reactor

        for pid, col in zip(palm_ids, colors):
            pt = lm[pid.value]
            px, py = int(pt.x * w), int(pt.y * h)

            # Core circle
            cv2.circle(frame, (px, py), 18, col, -1)
            cv2.circle(frame, (px, py), 24, (255, 255, 255), 2)

            # Pulse ring
            pulse_r = 30 + int(10 * math.sin(self.frame_count * 0.3))
            cv2.circle(frame, (px, py), pulse_r, col, 1)

            # Beam upward
            for seg in range(0, 120, 8):
                bx = px + random.randint(-3, 3)
                by = py - seg
                radius = max(1, 6 - seg // 20)
                alpha_val = 1.0 - seg / 120.0
                bcol = tuple(int(c * alpha_val) for c in col)
                if 0 < by < h:
                    cv2.circle(frame, (bx, by), radius, bcol, -1)

            # Particles
            self._add_particles(px, py, col, n=5, speed=4, life=25, size=3,
                                directed=True, dx=0, dy=-1)

    # ── Thor Lightning ──
    def _draw_lightning(self, frame, x1, y1, x2, y2, col, jitter=20, segs=10):
        pts = [(x1, y1)]
        for i in range(1, segs):
            t = i / segs
            mx = int(x1 + (x2 - x1) * t + random.randint(-jitter, jitter))
            my = int(y1 + (y2 - y1) * t + random.randint(-jitter, jitter))
            pts.append((mx, my))
        pts.append((x2, y2))
        for i in range(len(pts) - 1):
            cv2.line(frame, pts[i], pts[i+1], (200, 200, 255), 1)
            cv2.line(frame, pts[i], pts[i+1], col, 2)

    def thor_lightning(self, frame, lm, w, h):
        left_wrist = lm[mp.solutions.pose.PoseLandmark.LEFT_WRIST.value]
        right_wrist = lm[mp.solutions.pose.PoseLandmark.RIGHT_WRIST.value]
        lx, ly = int(left_wrist.x * w), int(left_wrist.y * h)
        rx, ry = int(right_wrist.x * w), int(right_wrist.y * h)

        # Lightning between hands
        for _ in range(3):
            self._draw_lightning(frame, lx, ly, rx, ry,
                                 (255, 200, 50), jitter=18, segs=12)

        # Arcs from each hand to corners
        col = (255, 220, 80)
        for _ in range(2):
            self._draw_lightning(frame, lx, ly, 0, random.randint(0, h),
                                 col, jitter=25, segs=10)
            self._draw_lightning(frame, rx, ry, w, random.randint(0, h),
                                 col, jitter=25, segs=10)

        # Glow at hands
        for cx, cy in [(lx, ly), (rx, ry)]:
            cv2.circle(frame, (cx, cy), 20, (255, 220, 100), -1)
            cv2.circle(frame, (cx, cy), 30, (180, 160, 50), 2)
            self._add_particles(cx, cy, (255, 220, 50), n=6, speed=5, life=20, size=3)

    # ── Captain America Shield ──
    def captain_shield(self, frame, lm, w, h):
        cx = w // 2
        cy = h // 2

        # Shield rings
        radii   = [120, 95, 60, 25]
        colors  = [(0, 0, 180), (200, 200, 200), (0, 0, 180), (200, 30, 30)]
        overlay = frame.copy()
        for r, c in zip(radii, colors):
            cv2.circle(overlay, (cx, cy), r, c, -1)
        # Star
        pts = []
        for i in range(5):
            outer_angle = math.pi / 2 + i * 2 * math.pi / 5
            inner_angle = outer_angle + math.pi / 5
            pts.append((int(cx + 20 * math.cos(outer_angle)),
                        int(cy - 20 * math.sin(outer_angle))))
            pts.append((int(cx + 10 * math.cos(inner_angle)),
                        int(cy - 10 * math.sin(inner_angle))))
        star = np.array(pts, np.int32)
        cv2.fillPoly(overlay, [star], (240, 240, 255))
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)

        # Shield glow edge
        pulse = int(5 * math.sin(self.frame_count * 0.2))
        cv2.circle(frame, (cx, cy), 122 + pulse, (150, 150, 255), 3)

    # ── Hulk Shockwave ──
    def hulk_smash(self, frame, lm, w, h):
        shoulder = lm[mp.solutions.pose.PoseLandmark.LEFT_SHOULDER.value]
        cx = int(shoulder.x * w)
        cy = h - 40  # Near bottom

        for ring in range(4):
            radius = (self.frame_count * 8 + ring * 40) % 300
            alpha = max(0, 1.0 - radius / 300.0)
            col = (int(0), int(200 * alpha), int(50 * alpha))
            thick = max(1, int(4 * alpha))
            cv2.ellipse(frame, (cx, cy), (radius, radius // 3),
                        0, 0, 360, col, thick)

        # Green particles explosion
        if self.frame_count % 3 == 0:
            self._add_particles(cx, cy, (50, 255, 80),
                                n=12, speed=8, life=35, size=6)

        # Ground crack lines
        for _ in range(5):
            angle = random.uniform(-math.pi / 4, math.pi + math.pi / 4)
            length = random.randint(50, 180)
            ex = int(cx + length * math.cos(angle))
            ey = int(cy + length * math.sin(angle) * 0.3)
            cv2.line(frame, (cx, cy), (ex, ey), (30, 180, 60), 2)

    # ── Spider-Man Web ──
    def spiderman_web(self, frame, lm, w, h):
        right_wrist = lm[mp.solutions.pose.PoseLandmark.RIGHT_WRIST.value]
        wx, wy = int(right_wrist.x * w), int(right_wrist.y * h)

        # Add to trail
        self.trail_points.append((wx, wy, self.frame_count))
        # Keep only recent trail
        self.trail_points = [(x, y, t) for x, y, t in self.trail_points
                             if self.frame_count - t < 25]

        # Draw web strands
        for i in range(len(self.trail_points) - 1):
            x1, y1, t1 = self.trail_points[i]
            x2, y2, t2 = self.trail_points[i + 1]
            age = self.frame_count - t1
            alpha = max(0, 1.0 - age / 25.0)
            col = (int(220 * alpha), int(220 * alpha), int(220 * alpha))
            thick = max(1, int(3 * alpha))
            cv2.line(frame, (x1, y1), (x2, y2), col, thick)

        # Web anchor lines
        for corner in [(0, 0), (w, 0), (0, h), (w, h)]:
            dist = math.hypot(wx - corner[0], wy - corner[1])
            if dist < 300:
                alpha = 1.0 - dist / 300.0
                col = (int(200 * alpha), int(200 * alpha), int(200 * alpha))
                mid_x = (wx + corner[0]) // 2 + random.randint(-20, 20)
                mid_y = (wy + corner[1]) // 2 + random.randint(30, 60)
                cv2.line(frame, (wx, wy), (mid_x, mid_y), col, 1)
                cv2.line(frame, (mid_x, mid_y), corner, col, 1)

        # Wrist glow
        cv2.circle(frame, (wx, wy), 12, (200, 200, 255), -1)
        cv2.circle(frame, (wx, wy), 18, (150, 150, 200), 2)
        self._add_particles(wx, wy, (200, 200, 255), n=3, speed=3, life=15, size=2)

    # ── Update & Draw All Particles ──
    def update_particles(self, frame):
        alive = []
        for p in self.particles:
            p.update()
            if p.life > 0:
                p.draw(frame)
                alive.append(p)
        self.particles = alive

    # ── HUD Overlay ──
    def draw_hud(self, frame, mode, fps, recording):
        h, w = frame.shape[:2]
        hero_colors = {
            'IRON MAN':  (0, 140, 255),
            'THOR':      (255, 220, 50),
            'CAP':       (60, 60, 220),
            'HULK':      (60, 220, 60),
            'SPIDER-MAN':(180, 50, 50),
            'IDLE':      (180, 180, 180),
        }
        col = hero_colors.get(mode, (200, 200, 200))

        # Top bar
        cv2.rectangle(frame, (0, 0), (w, 50), (10, 10, 10), -1)
        cv2.putText(frame, f'HERO: {mode}', (15, 33),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, col, 2)
        cv2.putText(frame, f'FPS: {fps:.0f}', (w - 120, 33),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)

        if recording:
            cv2.circle(frame, (w - 20, 25), 8, (0, 0, 255), -1)
            cv2.putText(frame, 'REC', (w - 70, 32),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        # Bottom controls
        cv2.rectangle(frame, (0, h - 40), (w, h), (10, 10, 10), -1)
        hints = '[S] Screenshot  [R] Record  [Q] Quit'
        cv2.putText(frame, hints, (15, h - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (160, 160, 160), 1)

print('✅ SuperheroFX engine ready!')

✅ SuperheroFX engine ready!


## Step 4: Pose Detector + Action Classifier

In [12]:
import mediapipe as mp
print(mp.__version__)
print(hasattr(mp, "solutions"))

0.10.9
True


In [13]:

mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

def classify_action(lm, w, h):
    """
    Rule-based action classifier using pose landmarks.
    Returns: 'IRON MAN' | 'THOR' | 'CAP' | 'HULK' | 'SPIDER-MAN' | 'IDLE'
    """
    def pt(pid):
        p = lm[pid.value]
        return np.array([p.x * w, p.y * h])

    nose       = pt(mp_pose.PoseLandmark.NOSE)
    l_shoulder = pt(mp_pose.PoseLandmark.LEFT_SHOULDER)
    r_shoulder = pt(mp_pose.PoseLandmark.RIGHT_SHOULDER)
    l_elbow    = pt(mp_pose.PoseLandmark.LEFT_ELBOW)
    r_elbow    = pt(mp_pose.PoseLandmark.RIGHT_ELBOW)
    l_wrist    = pt(mp_pose.PoseLandmark.LEFT_WRIST)
    r_wrist    = pt(mp_pose.PoseLandmark.RIGHT_WRIST)
    l_hip      = pt(mp_pose.PoseLandmark.LEFT_HIP)
    r_hip      = pt(mp_pose.PoseLandmark.RIGHT_HIP)

    shoulder_width = abs(l_shoulder[0] - r_shoulder[0]) + 1
    body_height    = abs(nose[1] - l_hip[1]) + 1

    # Both hands above head → IRON MAN
    if l_wrist[1] < nose[1] - 30 and r_wrist[1] < nose[1] - 30:
        return 'IRON MAN'

    # Arms wide open horizontally → THOR
    arm_span = abs(l_wrist[0] - r_wrist[0])
    wrist_y_diff = abs(l_wrist[1] - r_wrist[1])
    if arm_span > shoulder_width * 2.2 and wrist_y_diff < body_height * 0.25:
        return 'THOR'

    # Arms crossed (wrists close, near chest) → CAP
    wrist_dist = np.linalg.norm(l_wrist - r_wrist)
    chest_y = (l_shoulder[1] + r_shoulder[1]) / 2
    wrists_y_avg = (l_wrist[1] + r_wrist[1]) / 2
    if wrist_dist < shoulder_width * 0.8 and abs(wrists_y_avg - chest_y) < body_height * 0.3:
        return 'CAP'

    # Both hands low (below hips) → HULK
    hip_y = (l_hip[1] + r_hip[1]) / 2
    if l_wrist[1] > hip_y + 20 and r_wrist[1] > hip_y + 20:
        return 'HULK'

    # Right hand forward + left hand back → SPIDER-MAN
    r_wrist_lm = lm[mp_pose.PoseLandmark.RIGHT_WRIST.value]
    l_wrist_lm = lm[mp_pose.PoseLandmark.LEFT_WRIST.value]
    if r_wrist_lm.z < -0.1 and l_wrist_lm.z > 0:
        return 'SPIDER-MAN'

    return 'IDLE'

print('✅ Action classifier ready!')

✅ Action classifier ready!


## Step 5: 🚀 Launch the Real-Time System
> **Press Q to quit | S to screenshot | R to toggle recording**

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

In [17]:
def draw_ironman_beam(frame, wrist):
    x, y = int(wrist[0]), int(wrist[1])
    cv2.line(frame, (x, y), (x, y-200), (0, 0, 255), 8)
    cv2.circle(frame, (x, y-200), 20, (0, 0, 255), -1)

def glow_effect(frame):
    blur = cv2.GaussianBlur(frame, (15,15), 0)
    return cv2.addWeighted(frame, 1.2, blur, -0.2, 0)

In [15]:
def run_superhero_system(camera_index=0):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print('❌ Cannot open webcam. Try camera_index=1 or check permissions.')
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    fx = SuperheroFX()
    recording = False
    writer = None
    prev_time = time.time()
    current_mode = 'IDLE'
    mode_smoothing = []   # Buffer for stable mode detection

    with mp_pose.Pose(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        model_complexity=1
    ) as pose:

        print('🎬 System running! Look at the OpenCV window.')
        print('Controls: [S] Screenshot  [R] Record/Stop  [Q] Quit')

        while True:
            ret, frame = cap.read()
            if not ret:
                print('⚠️ Frame capture failed.')
                break

            frame = cv2.flip(frame, 1)  # Mirror
            h, w = frame.shape[:2]
            fx.frame_count += 1

            # FPS
            now = time.time()
            fps = 1.0 / max(now - prev_time, 0.001)
            prev_time = now

            # Pose estimation
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results = pose.process(rgb)
            rgb.flags.writeable = True

            if results.pose_landmarks:
                lm = results.pose_landmarks.landmark

                # Stable mode detection (majority vote over 5 frames)
                detected = classify_action(lm, w, h)
                mode_smoothing.append(detected)
                if len(mode_smoothing) > 5:
                    mode_smoothing.pop(0)
                from collections import Counter
                current_mode = Counter(mode_smoothing).most_common(1)[0][0]

                # Apply effects
                if current_mode == 'IRON MAN':
                    fx.iron_man_blast(frame, lm, w, h)
                elif current_mode == 'THOR':
                    fx.thor_lightning(frame, lm, w, h)
                elif current_mode == 'CAP':
                    fx.captain_shield(frame, lm, w, h)
                elif current_mode == 'HULK':
                    fx.hulk_smash(frame, lm, w, h)
                elif current_mode == 'SPIDER-MAN':
                    fx.spiderman_web(frame, lm, w, h)

                # Skeleton (subtle)
                mp_draw.draw_landmarks(
                    frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                    mp_draw.DrawingSpec(color=(80, 80, 80), thickness=1, circle_radius=2),
                    mp_draw.DrawingSpec(color=(60, 60, 60), thickness=1)
                )
            else:
                current_mode = 'IDLE'

            # Update particles
            fx.update_particles(frame)

            # HUD
            fx.draw_hud(frame, current_mode, fps, recording)

            # Recording
            if recording and writer:
                writer.write(frame)

            # Display
            cv2.imshow('🦸 Superhero FX System', frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('s'):
                ts = datetime.now().strftime('%Y%m%d_%H%M%S')
                path = f'screenshots/superhero_{ts}.jpg'
                cv2.imwrite(path, frame)
                print(f'📸 Screenshot saved: {path}')
            elif key == ord('r'):
                if not recording:
                    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
                    path = f'recordings/superhero_{ts}.avi'
                    fourcc = cv2.VideoWriter_fourcc(*'XVID')
                    writer = cv2.VideoWriter(path, fourcc, 20.0, (w, h))
                    recording = True
                    print(f'🔴 Recording started: {path}')
                else:
                    recording = False
                    if writer:
                        writer.release()
                        writer = None
                    print('⏹️ Recording stopped and saved.')

    cap.release()
    if writer:
        writer.release()
    cv2.destroyAllWindows()
    print('✅ System closed.')


# ─── RUN ───
# Change camera_index=1 if 0 doesn't work (e.g., on some laptops)
run_superhero_system(camera_index=0)

🎬 System running! Look at the OpenCV window.
Controls: [S] Screenshot  [R] Record/Stop  [Q] Quit


KeyboardInterrupt: 

---
## 🎮 Gesture Guide

| Pose | Effect | How To Do It |
|------|--------|-------------|
| 🔵 **Iron Man** | Energy blasts from palms | Raise **both hands above your head** |
| ⚡ **Thor** | Lightning arcs | Stretch **both arms wide** horizontally |
| 🛡️ **Captain America** | Force shield | **Cross arms** at chest level |
| 💚 **Hulk** | Ground shockwave | Drop **both hands below hips** |
| 🕷️ **Spider-Man** | Web shooting | Push **right hand forward** (toward cam), keep left hand back |

## 📁 Outputs
- Screenshots → `screenshots/` folder
- Videos → `recordings/` folder

## 🛠️ Troubleshooting
- **No webcam?** Try `camera_index=1` in `run_superhero_system(1)`
- **Slow FPS?** Set `model_complexity=0` in the Pose constructor
- **Window doesn't open?** Make sure you're running Jupyter locally (not in JupyterHub or cloud)